In [1]:
!pip install cryptography

In [2]:
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
from cryptography.hazmat.primitives import padding
from cryptography.hazmat.backends import default_backend
import os
import binascii

In [3]:
#  Generate 32-byte AES Key

def key_gen():
    return os.urandom(32).hex()

In [4]:
#  Encrypt (AES-256-CBC)

def encrypt(plain_text, encryption_key):
    key = binascii.unhexlify(encryption_key)  # Convert hex → bytes
    iv = os.urandom(16)

    # Pad plaintext to 16-byte block size
    padder = padding.PKCS7(128).padder()
    padded_data = padder.update(plain_text.encode()) + padder.finalize()

    cipher = Cipher(algorithms.AES(key), modes.CBC(iv), backend=default_backend())
    encryptor = cipher.encryptor()
    encrypted_data = encryptor.update(padded_data) + encryptor.finalize()

    return iv.hex() + ":" + encrypted_data.hex()

In [5]:
#  Decrypt (AES-256-CBC)

def decrypt(cipher_text, encryption_key):
    iv_hex, encrypted_hex = cipher_text.split(":")
    iv = binascii.unhexlify(iv_hex)
    encrypted_data = binascii.unhexlify(encrypted_hex)
    key = binascii.unhexlify(encryption_key)

    cipher = Cipher(algorithms.AES(key), modes.CBC(iv), backend=default_backend())
    decryptor = cipher.decryptor()
    decrypted_padded = decryptor.update(encrypted_data) + decryptor.finalize()

    # Unpad plaintext
    unpadder = padding.PKCS7(128).unpadder()
    decrypted = unpadder.update(decrypted_padded) + unpadder.finalize()

    return decrypted.decode()


In [7]:
# Generate and print a key 
key = key_gen()
print("Generated Key:", key)

cipher = encrypt("Hello, this is a secret message!", key)
print("Encrypted:", cipher)

plain = decrypt(cipher, key)
print("Decrypted:", plain)

Generated Key: 52257b98839d6640404851d6f6dd6591ec92dcc502a07046d68bc6495faaa7b4
Encrypted: 34adfae76770f831ffdcf685dc72019e:06d62230b069c231ef7eb118c50ed6f3c80128278be11b577e28340b272b04cdddb4baa75288479293150b1ec02017f1
Decrypted: Hello, this is a secret message!
